# uGMRT Preprocess Workflow (Production)

This notebook is a thin workflow wrapper around module functions in `ugmrt_query.py`.

Core loop:
1. Derive bandpass
2. Run diagnostics
3. Propose/write flag table updates
4. Repeat from Step 1 with one or more flag tables

All operations are non-destructive: input visibilities are never modified on disk.

In [7]:
import importlib
import sys
from pathlib import Path

if 'ugmrt_query' in sys.modules:
    importlib.reload(sys.modules['ugmrt_query'])
import ugmrt_query as q

print('ugmrt_query loaded')

ugmrt_query loaded


In [9]:
# User configuration
BASE_DIR = Path('/media/wasim/gmrt_40_014')
WORK_DIR = BASE_DIR / 'work'

CAL_FITS = WORK_DIR / '40_014_25JUL2021_copy' / '40_014_25jul2021_gsb.FITS'
INDEX_CACHE = WORK_DIR / '40_014_25jul2021_gsb.row_index_cache.npz'

# Row-index cache validation mode:
#   'fast'      -> compare file size + modification time
#   'fast+sha'  -> fast check first; on mismatch, verify SHA before rebuild
#   'sha256'    -> compare full SHA256 hash every run (strict, slower)
#   'none'      -> trust cache without checking source file
INDEX_VALIDATION_MODE = 'fast+sha'

# Multiple on-disk flag tables can be merged on-the-fly
FLAG_TABLE_BASE = WORK_DIR / '3c48_flag_table.json'
FLAG_TABLE_SESSION = WORK_DIR / '3c48_flag_table_session.json'
FLAG_TABLE_PATHS = [p for p in [FLAG_TABLE_BASE, FLAG_TABLE_SESSION] if p.exists()]

# In-memory flag tables proposed in earlier dry-run iterations can also be reused on-the-fly.
if 'PENDING_FLAG_TABLES' not in globals():
    PENDING_FLAG_TABLES = []

# User control: set USE_PENDING_FLAG_TABLES=False if you do not want dry-run proposals
# from previous iterations to affect the next bandpass solve.
USE_PENDING_FLAG_TABLES = True

# Optional reset switch: set CLEAR_PENDING_FLAG_TABLES=True once to discard any previously
# carried in-memory proposals, then run this cell.
CLEAR_PENDING_FLAG_TABLES = False
if CLEAR_PENDING_FLAG_TABLES:
    PENDING_FLAG_TABLES = []

ITER_TAG = 'iter01'
DRY_RUN_BANDPASS = True
DRY_RUN_FLAG_WRITE = True

# Solve options
SOURCE = '3C48'
STOKES = ('RR', 'LL')
CHAN_RANGE = (64, 191)
MAX_ROWS_SOLVE = 150_000
SMOOTH_WINDOW = 5
MIN_BASELINES = 20

# Diagnostics options
EXCLUDE_FOR_PLOTS = []
SKIP_EDGE_CHANNELS = (0, 0)
TOP_N = 12

print('CAL_FITS:', CAL_FITS)
print('INDEX_CACHE:', INDEX_CACHE)
print('INDEX_VALIDATION_MODE:', INDEX_VALIDATION_MODE)
print('FLAG_TABLE_PATHS:', FLAG_TABLE_PATHS)
print('Pending in-memory flag tables available:', len(PENDING_FLAG_TABLES))
print('USE_PENDING_FLAG_TABLES:', USE_PENDING_FLAG_TABLES)
print('CLEAR_PENDING_FLAG_TABLES:', CLEAR_PENDING_FLAG_TABLES)
print('ITER_TAG:', ITER_TAG)
print('DRY_RUN_BANDPASS:', DRY_RUN_BANDPASS)
print('DRY_RUN_FLAG_WRITE:', DRY_RUN_FLAG_WRITE)

CAL_FITS: /media/wasim/gmrt_40_014/work/40_014_25JUL2021_copy/40_014_25jul2021_gsb.FITS
INDEX_CACHE: /media/wasim/gmrt_40_014/work/40_014_25jul2021_gsb.row_index_cache.npz
INDEX_VALIDATION_MODE: fast+sha
FLAG_TABLE_PATHS: []
Pending in-memory flag tables available: 0
USE_PENDING_FLAG_TABLES: True
CLEAR_PENDING_FLAG_TABLES: False
ITER_TAG: iter01
DRY_RUN_BANDPASS: True
DRY_RUN_FLAG_WRITE: True


In [11]:
# Step 1: Load or build persistent row index cache
row_index = q.get_or_build_row_index(
    CAL_FITS,
    cache_path=INDEX_CACHE,
    force_rebuild=False,
    validation_mode=INDEX_VALIDATION_MODE,
    write_cache=True,
 )

print('Index path:', row_index.get('index_cache_path', INDEX_CACHE))
print('Source identity:', row_index.get('source_identity'))
print('Source SHA256:', row_index.get('source_sha256'))

Loaded row index cache: /media/wasim/gmrt_40_014/work/40_014_25jul2021_gsb.row_index_cache.npz
Index path: /media/wasim/gmrt_40_014/work/40_014_25jul2021_gsb.row_index_cache.npz
Source identity: {'size_bytes': 8107850880, 'mtime_ns': 1627291136746819895}
Source SHA256: 783ba5db6cbe496ec870b1a83ea55d89d28357a3ebe04f35b0231ec2b62213c6


In [12]:
# Step 2: Derive bandpass (non-destructive)
BANDPASS_OUT_BASE = WORK_DIR / '3c48_bandpass_25jul_gsb.npz'
active_pending_flag_tables = PENDING_FLAG_TABLES if USE_PENDING_FLAG_TABLES else []

bandpass_run = q.derive_bandpass_iteration(
    fits_path=CAL_FITS,
    index=row_index,
    bandpass_out=BANDPASS_OUT_BASE,
    source=SOURCE,
    stokes=STOKES,
    chan_range=CHAN_RANGE,
    max_rows=MAX_ROWS_SOLVE,
    smooth_window=SMOOTH_WINDOW,
    min_baselines=MIN_BASELINES,
    ignore_autos=True,
    flag_table_path=FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None,
    flag_table=active_pending_flag_tables if active_pending_flag_tables else None,
    iteration_tag=ITER_TAG,
    dry_run=DRY_RUN_BANDPASS,
 )

bandpass_sol = bandpass_run['solution']
bandpass_out_path = bandpass_run['bandpass_out']

print('Bandpass dry-run:', bandpass_run['dry_run'])
print('Bandpass output path:', bandpass_out_path)
print('On-disk flag tables used:', FLAG_TABLE_PATHS)
print('Pending in-memory flag tables available:', len(PENDING_FLAG_TABLES))
print('Pending in-memory flag tables applied:', len(active_pending_flag_tables))
print('Merged flag tables seen by solve:', bandpass_sol.get('flag_table_paths', []))
print('Merged flag table count:', bandpass_sol.get('flag_table_count', 0))
print('Rows dropped by merged flags:', bandpass_sol.get('solve_dropped_rows_by_flag_table', 0))

[load_vis] 57,456 rows, 353 MB read, 15.6s
Bandpass dry-run: True
Bandpass output path: /media/wasim/gmrt_40_014/work/3c48_bandpass_25jul_gsb_iter01.npz
On-disk flag tables used: []
Pending in-memory flag tables available: 0
Pending in-memory flag tables applied: 0
Merged flag tables seen by solve: []
Merged flag table count: 0
Rows dropped by merged flags: 0


In [ ]:
# Step 3: Diagnostics + Step 4: Propose flags + Step 5: Update flag table
DIAG_PLOT_BASE = WORK_DIR / '3c48_bandpass_diagnostics.png'
diag_plot_path = q.tagged_output_path(DIAG_PLOT_BASE, ITER_TAG)

diag = q.run_bandpass_diagnostics(
    row_index,
    bandpass_sol,
    source=SOURCE,
    chan_range=CHAN_RANGE,
    stokes=STOKES,
    max_rows=60_000,
    exclude_antennas=EXCLUDE_FOR_PLOTS,
    skip_edge_channels=SKIP_EDGE_CHANNELS,
    top_n=TOP_N,
    title=f'3C48 diagnostics | {ITER_TAG}',
    save_path=diag_plot_path,
 )

proposal = q.propose_flag_updates_from_diagnostics(
    diag,
    pol='LL',
    mode='baselines',
    antenna_threshold_jy=180.0,
    baseline_threshold_jy=800.0,
    max_add_antennas=4,
    max_add_baselines=6,
 )

print('Candidate antennas:', proposal['candidate_antennas'])
print('Candidate baselines:', proposal['candidate_baselines'])

flag_update = q.update_flag_table(
    output_path=FLAG_TABLE_SESSION,
    add_antennas=proposal['proposal']['bad_antennas'],
    add_baselines=proposal['proposal']['bad_baselines'],
    base_flag_table_paths=[FLAG_TABLE_BASE] if FLAG_TABLE_BASE.exists() else None,
    notes=f'Auto-proposed from diagnostics {ITER_TAG}',
    dry_run=DRY_RUN_FLAG_WRITE,
 )

# Even in dry-run mode, keep the merged proposed table in memory so the next iteration
# can use it on-the-fly without writing anything to disk.
if DRY_RUN_FLAG_WRITE:
    PENDING_FLAG_TABLES = [flag_update['flag_table']]
else:
    PENDING_FLAG_TABLES = []

print('Flag update dry-run:', flag_update['dry_run'])
print('Flag table target:', flag_update['output_path'])
print('Would add antennas:', flag_update['added_antennas'])
print('Would add baselines:', flag_update['added_baselines'])
print('Pending in-memory flag tables for next iteration:', len(PENDING_FLAG_TABLES))

## Iteration Loop
For the next iteration:
1. Set `ITER_TAG` to the next value (for example `iter02`).
2. If you want outputs written, set `DRY_RUN_BANDPASS=False` and `DRY_RUN_FLAG_WRITE=False`.
3. Re-run cells 4 to 6.
4. On-disk flag tables are always taken from `FLAG_TABLE_PATHS`.
5. Dry-run proposals are kept in memory as `PENDING_FLAG_TABLES`, but they are only applied if `USE_PENDING_FLAG_TABLES=True`.
6. If you want to stop propagation completely, set `USE_PENDING_FLAG_TABLES=False`.
7. If you want to discard already-carried in-memory proposals, set `CLEAR_PENDING_FLAG_TABLES=True` once and rerun the configuration cell.

All filtering and calibration are performed in memory; raw vis data on disk remains untouched.